# Solar AI — ground-PV CONDITION/DEFECT research detector (YOLO11n)

Trains `condition_yolo11n_baseline_v1` on a Kaggle GPU. **5-class second-stage condition model:**
`0=CLEAN 1=DUSTY 2=BIRD_DROP 3=PHYSICAL_DAMAGE 4=ELECTRICAL_DAMAGE`.

- **Experimental research model only.** NOT the panel detector. Does NOT touch / replace
  the production aerial detector `weights/yolo_solar.pt` or the ground panel detector
  `weights/yolo_solar_ground_research_v1.pt`.
- Base model: pretrained **YOLO11n** COCO checkpoint (never a Solar AI model).
- Dataset: `edithstark/solar-ai-ground-condition-research-v1` (PRIVATE). Built from Roboflow
  S1 `defects-kgig6/detection-for-defects-in-solar-panels` v2 + S3
  `solar-panel-a5cyn/dusty-and-bird-droppings` v1, de-duplicated, leakage-safe split.

## ⚠️ PROVENANCE — grade C, UNVERIFIED
The underlying images ultimately trace to Kaggle `pythonafroz/solar-panel-images`
(licence **Unknown**). Tag: `personal-experimental-unverified-provenance`. Do NOT
redistribute / publish / use commercially / claim verified CC BY image rights.

## ⚠️ ELECTRICAL_DAMAGE — HIGH SEMANTIC RISK
Every ELECTRICAL_DAMAGE annotation is S1's `electrical-damage` class. The provenance
audit found these boxes visually correspond to **cell cracks**, not confirmed electrical
failure. Labels are preserved unchanged (not relabelled). A high ELECTRICAL_DAMAGE score
is **not** evidence of electrical-fault diagnosis. Metrics for this class are reported
separately and prominently.

## CLEAN — semantic limitation
CLEAN is trained only where the source carried explicit clean-region annotations. Absence
of a defect detection is **not** proof an image/panel is Clean. No CLEAN boxes were
fabricated.

imgsz=640, epochs=100, patience=20, seed=42, cos_lr, optimizer=auto. Attach the dataset
in the sidebar and set Accelerator = GPU before running.


## 1. Environment


In [ ]:
import sys, platform, subprocess, json, os, time, hashlib, shutil
from pathlib import Path
print('Python :', platform.python_version())


### CUDA — fail closed if unavailable
Kaggle's free GPU pool can hand out a Pascal **Tesla P100 (sm_60)**. Pin
`torch==2.7.1` / `torchvision==0.22.1` (CUDA 12.6 wheels, last line retaining Pascal
support) — the pins proven on this account's prior ground-detector run.


In [ ]:
PINNED = ['torch==2.7.1', 'torchvision==0.22.1', 'ultralytics==8.4.138', 'PyYAML>=6.0.1']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINNED], check=True)

import torch
assert torch.cuda.is_available(), 'CUDA unavailable — set Settings -> Accelerator -> GPU.'
import torchvision, ultralytics
ENV = dict(
    python=platform.python_version(), torch=torch.__version__,
    torchvision=torchvision.__version__, cuda=torch.version.cuda,
    ultralytics=ultralytics.__version__,
    gpu=torch.cuda.get_device_name(0),
    gpu_capability='.'.join(map(str, torch.cuda.get_device_capability(0))),
    gpu_memory_gb=round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
)
print(json.dumps(ENV, indent=2))


## 2. Locate the attached Kaggle dataset + verify provenance


In [ ]:
candidates = list(Path('/kaggle/input').rglob('data.yaml'))
assert candidates, 'No data.yaml under /kaggle/input — attach the dataset in the sidebar.'
SRC_DATA_YAML = None
for c in candidates:
    if 'condition' in str(c).lower():
        SRC_DATA_YAML = c; break
SRC_DATA_YAML = SRC_DATA_YAML or candidates[0]
DATA_ROOT = SRC_DATA_YAML.parent
print('mounted data.yaml :', SRC_DATA_YAML)
print(SRC_DATA_YAML.read_text())

import yaml as _yaml
_cfg = _yaml.safe_load(SRC_DATA_YAML.read_text())
_cfg['path'] = str(DATA_ROOT.resolve())
_cfg['train'], _cfg['val'], _cfg['test'] = 'images/train', 'images/valid', 'images/test'
DATA_YAML = Path('/kaggle/working/ground_condition_data.yaml')
DATA_YAML.write_text(_yaml.safe_dump(_cfg, sort_keys=False))
print('training data.yaml (absolute):'); print(DATA_YAML.read_text())
for _s in ('images/train','images/valid','images/test','labels/train','labels/valid','labels/test'):
    assert (DATA_ROOT / _s).is_dir(), f'missing {_s} under {DATA_ROOT}'
print('all split dirs present')

# provenance shipped inside the dataset
PROV = DATA_ROOT / 'provenance'
assert PROV.is_dir(), 'no provenance/ directory in the dataset'
pkg_summary = json.loads((DATA_ROOT / 'PACKAGE_SUMMARY.json').read_text())
print('provenance_grade :', pkg_summary['provenance_grade'])
print('provenance_tag   :', pkg_summary['provenance_tag'])
print('electrical_damage_semantic_risk :', pkg_summary['electrical_damage_semantic_risk']['flag'])
assert pkg_summary['provenance_grade'] == 'C'
assert pkg_summary['provenance_tag'] == 'personal-experimental-unverified-provenance'
assert pkg_summary['electrical_damage_semantic_risk']['flag'] == 'HIGH'
assert pkg_summary['no_family_crosses_splits'] is True

def _sha_file(p):
    h = hashlib.sha256(); h.update(Path(p).read_bytes()); return h.hexdigest()
SPLIT_CSV = PROV / 'condition_split_manifest.csv'
TAXO_JSON = PROV / 'condition_taxonomy.json'
DATASET_HASHES = dict(
    split_manifest_sha256=_sha_file(SPLIT_CSV),
    taxonomy_sha256=_sha_file(TAXO_JSON),
    package_dataset_content_sha256=pkg_summary['dataset_content_sha256'],
    package_data_yaml_sha256=pkg_summary['data_yaml_sha256'],
)
print(json.dumps(DATASET_HASHES, indent=2))
assert DATASET_HASHES['split_manifest_sha256'] == pkg_summary['split_manifest_sha256']
assert DATASET_HASHES['taxonomy_sha256'] == pkg_summary['taxonomy_sha256']


## 3. Validate taxonomy + per-split, per-class statistics (no training yet)


In [ ]:
import yaml, csv
from collections import Counter, defaultdict
cfg = yaml.safe_load(DATA_YAML.read_text())
names = cfg['names']
if isinstance(names, list): names = {i: n for i, n in enumerate(names)}
NAMES = {int(k): v for k, v in names.items()}
EXPECTED = {0:'CLEAN', 1:'DUSTY', 2:'BIRD_DROP', 3:'PHYSICAL_DAMAGE', 4:'ELECTRICAL_DAMAGE'}
assert NAMES == EXPECTED, f'taxonomy mismatch: {NAMES} != {EXPECTED}'
assert 'panel' not in [v.lower() for v in NAMES.values()], 'PANEL must not be a class'
assert not any('snow' in v.lower() for v in NAMES.values())
print('taxonomy OK:', NAMES)

# unique source-family counts per class, per split, from the bundled split manifest
rows = list(csv.DictReader(open(SPLIT_CSV, encoding='utf-8')))
fam_by_split_class = defaultdict(lambda: defaultdict(set))
img_by_split_class = defaultdict(Counter)
for r in rows:
    s = r['derived_split']
    for c in r['unified_classes'].split(';'):
        if c and c != '(none)':
            fam_by_split_class[s][c].add(r['family_id'])
            img_by_split_class[s][c] += 1

STATS = {}
for split_key in ('train', 'val', 'test'):
    rel = cfg[split_key]; sname = {'train':'train','val':'valid','test':'test'}[split_key]
    img_dir = DATA_ROOT / rel
    lbl_dir = DATA_ROOT / rel.replace('images/', 'labels/')
    imgs = [p for p in img_dir.glob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
    inst, img_with = Counter(), Counter()
    for im in imgs:
        lp = lbl_dir / (im.stem + '.txt')
        if not lp.is_file(): continue
        seen = set()
        for line in lp.read_text().splitlines():
            p = line.split()
            if len(p) == 5:
                c = EXPECTED[int(p[0])]; inst[c] += 1; seen.add(c)
        for c in seen: img_with[c] += 1
    STATS[sname] = dict(
        images=len(imgs), instances=dict(inst), images_with_class=dict(img_with),
        unique_families_per_class={c: len(f) for c, f in fam_by_split_class[sname].items()},
    )
    print(f'  {sname:5s} images={len(imgs):5d} instances={dict(inst)}')
    print(f'        images_with={dict(img_with)}  fam={STATS[sname]["unique_families_per_class"]}')


## 4. Train `condition_yolo11n_baseline_v1`
Base `yolo11n.pt` (COCO-pretrained). Reproducible baseline, not leaderboard tuning.
Augmentation matches the prior ground-detector experiment (conservative — must not
distort defect semantics).


In [ ]:
from ultralytics import YOLO
PROJECT = '/kaggle/working/research_runs/ground_condition'
RUN_NAME = 'condition_yolo11n_baseline_v1'
BASE_MODEL = 'yolo11n.pt'
SEED = 42

train_args = dict(
    data=str(DATA_YAML), epochs=100, imgsz=640, patience=20, seed=SEED,
    batch=-1, device=0, project=PROJECT, name=RUN_NAME, exist_ok=True,
    pretrained=True, cos_lr=True, optimizer='auto', verbose=True, val=True,
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, degrees=5.0, translate=0.1, scale=0.3,
    shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.5, mosaic=0.5, mixup=0.0, copy_paste=0.0,
)
print(json.dumps({k: str(v) for k, v in train_args.items()}, indent=2))

t0 = time.time()
model = YOLO(BASE_MODEL)
results = model.train(**train_args)
train_seconds = time.time() - t0
RUN_DIR = Path(results.save_dir)
print('run dir:', RUN_DIR, ' elapsed(s):', round(train_seconds))


## 5. Evaluate — validation and the HELD-OUT leakage-clean test set, per class


In [ ]:
best = RUN_DIR / 'weights' / 'best.pt'
assert best.is_file(), best
eval_model = YOLO(str(best))

def evaluate(split):
    m = eval_model.val(data=str(DATA_YAML), split=split, imgsz=640, iou=0.5,
                       plots=True, verbose=False)
    box = m.box
    per_class = {}
    for pos, cidx in enumerate(list(box.ap_class_index)):
        p, r, ap50, ap = box.class_result(pos)
        per_class[EXPECTED[int(cidx)]] = dict(precision=float(p), recall=float(r),
                                              AP50=float(ap50), AP50_95=float(ap))
    return dict(
        overall=dict(precision=float(box.mp), recall=float(box.mr),
                     mAP50=float(box.map50), mAP50_95=float(box.map)),
        per_class=per_class, save_dir=str(m.save_dir))

evaluation = {}
for split in ('val', 'test'):
    evaluation[split] = evaluate(split)
    print('\n==', split, '==')
    print(json.dumps(evaluation[split]['overall'], indent=2))
    print('per-class:', json.dumps(evaluation[split]['per_class'], indent=2))

# merge in the test image / instance / family counts computed in cell 3
for c, d in evaluation['test']['per_class'].items():
    d['test_images_with_class'] = STATS['test']['images_with_class'].get(c, 0)
    d['test_instances'] = STATS['test']['instances'].get(c, 0)
    d['test_unique_families'] = STATS['test']['unique_families_per_class'].get(c, 0)

print('\n*** ELECTRICAL_DAMAGE — HIGH SEMANTIC RISK ***')
print('These metrics may simply reflect the source dataset labelling cell cracks as')
print('\'electrical-damage\'. A high score is NOT electrical-fault diagnosis.')
print(json.dumps(evaluation['test']['per_class'].get('ELECTRICAL_DAMAGE', {}), indent=2))


## 6. Package artifacts + training manifest + SHA-256


In [ ]:
OUT = Path('/kaggle/working/output'); OUT.mkdir(exist_ok=True)
for name in ('best.pt', 'last.pt'):
    src = RUN_DIR / 'weights' / name
    if src.is_file(): shutil.copy2(src, OUT / name)
for pat in ('results.csv','results.png','args.yaml',
            'confusion_matrix.png','confusion_matrix_normalized.png',
            'BoxPR_curve.png','BoxF1_curve.png','BoxP_curve.png','BoxR_curve.png',
            'PR_curve.png','F1_curve.png','P_curve.png','R_curve.png',
            'labels.jpg','labels_correlogram.jpg',
            'train_batch*.jpg','val_batch*.jpg'):
    for f in RUN_DIR.rglob(pat): shutil.copy2(f, OUT / f.name)
for sd in {e['save_dir'] for e in evaluation.values()}:
    sd = Path(sd)
    if sd.is_dir(): shutil.copytree(sd, OUT / ('eval_' + sd.name), dirs_exist_ok=True)

pred_dir = OUT / 'predictions'; pred_dir.mkdir(exist_ok=True)
test_imgs = sorted((DATA_ROOT / 'images/test').glob('*'))[:24]
if test_imgs:
    YOLO(str(best)).predict(source=[str(p) for p in test_imgs], conf=0.25, save=True,
                            project=str(pred_dir), name='test_examples', exist_ok=True)

sha = hashlib.sha256((OUT / 'best.pt').read_bytes()).hexdigest()
last_sha = hashlib.sha256((OUT / 'last.pt').read_bytes()).hexdigest() if (OUT/'last.pt').is_file() else None

import csv as _csv
results_rows = list(_csv.DictReader(open(RUN_DIR / 'results.csv')))
best_epoch = None
if results_rows:
    key = next((k for k in results_rows[0] if 'mAP50-95' in k and 'B' in k), None)
    if key:
        best_epoch = max(range(len(results_rows)), key=lambda i: float(results_rows[i][key])) + 1

manifest = dict(
    experiment_name=RUN_NAME,
    model_role='EXPERIMENTAL RESEARCH condition/defect detector (second stage, on panel crops) — NOT a panel detector, NOT production',
    dataset_kaggle='edithstark/solar-ai-ground-condition-research-v1',
    dataset_sources=['Roboflow defects-kgig6/detection-for-defects-in-solar-panels v2',
                     'Roboflow solar-panel-a5cyn/dusty-and-bird-droppings v1'],
    provenance_grade='C', provenance_tag='personal-experimental-unverified-provenance',
    provenance_warning='underlying images trace to Kaggle pythonafroz/solar-panel-images (licence Unknown); do NOT redistribute/publish/use commercially/claim verified CC BY',
    electrical_damage_semantic_risk='HIGH — S1 electrical-damage boxes visually correspond to cell cracks; label preserved, not relabelled; high score is NOT electrical-fault diagnosis',
    clean_semantic_limitation='CLEAN trained only on explicit clean-region annotations; absence of a defect detection is NOT proof of Clean; no CLEAN boxes fabricated',
    classes=EXPECTED,
    base_model=BASE_MODEL, pretrained_source='YOLO11n COCO-pretrained (Ultralytics)',
    epochs=train_args['epochs'], imgsz=train_args['imgsz'], batch=train_args['batch'],
    patience=train_args['patience'], optimizer='auto', cos_lr=True, seed=SEED,
    augmentation={k: train_args[k] for k in ('hsv_h','hsv_s','hsv_v','degrees','translate',
        'scale','shear','perspective','flipud','fliplr','mosaic','mixup','copy_paste')},
    full_train_args={k: str(v) for k, v in train_args.items()},
    device='cuda:0', env=ENV,
    dataset_hashes=DATASET_HASHES,
    dataset_stats=STATS,
    train_seconds=train_seconds, best_epoch=best_epoch, total_epochs=train_args['epochs'],
    run_dir=str(RUN_DIR),
    val_metrics=evaluation['val']['overall'], test_metrics=evaluation['test']['overall'],
    per_class=dict(val=evaluation['val']['per_class'], test=evaluation['test']['per_class']),
    checkpoint_sha256=sha, last_checkpoint_sha256=last_sha,
)
(OUT / 'training_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
(OUT / 'evaluation.json').write_text(json.dumps(evaluation, indent=2, default=str))
(OUT / 'environment.json').write_text(json.dumps(ENV, indent=2))
print('best.pt sha256:', sha)
print(json.dumps(manifest, indent=2, default=str))
print('\nArtifacts:', sorted(p.name for p in OUT.iterdir()))


## 7. Retrieve
Download `/kaggle/working/output/`. Locally save `best.pt` as
`weights/yolo_solar_condition_research_v1.pt` (gitignored research artifact) — **never**
overwrite `weights/yolo_solar.pt` or `weights/yolo_solar_ground_research_v1.pt`.

This condition model is a research artifact only. It is a **second-stage** model that
runs on panel crops. It is not a panel detector, makes no RGB-hotspot claim, and a high
ELECTRICAL_DAMAGE score is not an electrical-fault diagnosis (source labels look like
cell cracks).
